In [0]:
#Para el total nacional calcule (a Junio de 2026):
#población en edad de trabajar
#población que no está en edad de trabajar
#población económicamente activa
#población económicamente inactiva
#tasa de ocupación
#tasa de desempleo

In [0]:
#población en edad de trabajar
import pandas as pd
import requests
from io import StringIO
#Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")
ocupados.display()


# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()
ocupados= ocupados.toPandas()



In [0]:

cgse.head()


In [0]:

#El valor FEX_C18 es el factor que usamos para estimar la población representada por cada persona.
cgse.groupby('PERIODO')['FEX_C18'].agg(
    ['count', 'sum', 'mean']
)

In [0]:
# 1. Revisar el total de registros
print("Número de registros:", len(cgse))

# 2. Revisar los periodos
print(cgse['PERIODO'].value_counts().sort_index())

# 3. Sumar el factor de expansión
print("Suma de FEX_C18:", cgse['FEX_C18'].sum())

In [0]:
print(cgse['PT'].value_counts(dropna=False))

In [0]:
#POBLACIÓN TOTAL
print(cgse['FEX_C18'].describe())

PT = cgse['FEX_C18'].sum()
print(f"Población total expandida: {PT:,.0f}")

In [0]:
#POBLACIÓN EN EDAD PARA TRABAJAR
#PET
# Seleccionar personas de 15 años o más
pet = cgse[cgse['P6040'] >= 15]

# Estimar la población en edad de trabajar
PET = pet['FEX_C18'].sum()

print(f"Población en edad de trabajar: {PET:,.0f}")

#41,2 millones de personas en edad de trabajar.


In [0]:
#POBLACIÓN QUE NO ESTA EN EDAD PARA TRABAJAR

# Según lo leído en DANE profesor , 15 sale de la definición de la Población en Edad de Trabajar (PET) que utiliza el DANE en la GEIH

PNE = cgse[(cgse['P6040'] < 15)]['FEX_C18'].sum()
PNE
#Indicador: Población que no está en edad de trabajar (PNE).
#Condición: P6040 < 15.
#Factor de expansión: FEX_C18.



#Resultado estimado: 11.134.618 personas.

In [0]:
#POBLACIÓN ECONÓMICAMENTE ACTIVA
#PEA
PEA = ft[ft['FT'] == 1]['FEX_C18'].sum()
PEA

     

In [0]:
#POBLACION ECONOMICAMENTE INACTIVA 
PEI = PET - PEA
PEI
print(f'Población inactiva: {PEI:,.2f} personas')


In [0]:
#TASA OCUPADOS
PO = ocupados['FEX_C18'].sum()
TO=(PO/PET)*100
print(f'Tasa de ocupacion:  {TO:.2f} %')

#La tasa de ocupación fue del 59,4 %, lo que significa que, por cada 100 personas en edad de trabajar, aproximadamente 59 se encontraban ocupadas.
#TO (59,4 %): toma como referencia a las personas en edad de trabajar.


In [0]:
#TASA DESOCUPADOS
D = desempleados[desempleados['DSI'] == 1]['FEX_C18'].sum()
TD = (D / PEA)*100
print(f'Tasa de desocupacion:  {TD:.2f} %')

#Por cada 100 personas que hacen parte de la población económicamente activa (PEA), aproximadamente 8 están desocupadas, es decir, no tienen empleo y están buscando trabajo.
#TD (8 %): toma como referencia a las personas económicamente activas, es decir, ocupadas y desocupadas que buscan empleo.

In [0]:
#TODAS LAS RESPUESTAS PUNTO 1
print("Población desocupada (D):", D)
print("Población económicamente activa (PEA):", PEA)
print("Población económicamente inactiva (PEI):", PEI)
print("Población en edad de trabajar (PET):", PET)
print("Población ocupada (PO):", PO)
print("Tasa de desocupación (TD):", TD, "%")
print("Tasa de ocupación (TO):", TO, "%")
print("Población total (PT):", PT)
print("Población que no está en edad de trabajar (PNE):", PNE)



In [0]:
# PUNTO 2
# Tasas de desempleo de hombres y mujeres por departamento
# Junio de 2026

# Filtrar la PEA
pea_genero = ft[
    (ft['PERIODO'] == 20260626) &
    (ft['FT'] == 1)
].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

# Filtrar las personas desempleadas
desempleados_genero = desempleados[
    (desempleados['PERIODO'] == 20260626) &
    (desempleados['DSI'] == 1)
].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

# Crear la tabla
tabla_genero = pd.DataFrame({
    'PEA': pea_genero,
    'DESEMPLEADOS': desempleados_genero
})

tabla_genero['DESEMPLEADOS'] = (
    tabla_genero['DESEMPLEADOS'].fillna(0)
)

# Calcular la tasa de desempleo
tabla_genero['TD'] = (
    tabla_genero['DESEMPLEADOS'] /
    tabla_genero['PEA']
) * 100

# Convertir los índices en columnas
tabla_genero = tabla_genero.reset_index()

# Asignar nombres a los géneros
tabla_genero['Genero'] = tabla_genero['P3271'].map({
    1: 'Hombres',
    2: 'Mujeres'
})

# Asignar nombres a los departamentos
tabla_genero['Departamento'] = tabla_genero['DPTO'].map(dpto_nombre)

# Mostrar las tasas por género y departamento
print(tabla_genero[
    ['Departamento', 'Genero', 'PEA', 'DESEMPLEADOS', 'TD']
].sort_values(['Genero', 'TD'], ascending=[True, False]).to_string(index=False))

# Departamento con mayor tasa para cada género
mayores = tabla_genero.loc[
    tabla_genero.groupby('Genero')['TD'].idxmax()
]

print("\nDEPARTAMENTOS CON MAYOR TASA DE DESEMPLEO:")
print(mayores[
    ['Departamento', 'Genero', 'TD']
].to_string(index=False))

# Departamento con menor tasa para cada género
menores = tabla_genero.loc[
    tabla_genero.groupby('Genero')['TD'].idxmin()
]

print("\nDEPARTAMENTOS CON MENOR TASA DE DESEMPLEO:")
print(menores[
    ['Departamento', 'Genero', 'TD']
].to_string(index=False))

#: En junio de 2026, San Andrés presentó la mayor tasa de desempleo entre los hombres, con un 20,67 %, mientras que Arauca registró la mayor tasa entre las mujeres, con un 27,24 %. Por su parte, Vaupés presentó las tasas más bajas para ambos géneros, con un 0 %. Este último resultado debe verificarse, ya que podría deberse a la ausencia de registros de personas desempleadas en la muestra.


In [0]:
#PUNTO 3

import pandas as pd
import numpy as np

# 1. Filtrar junio de 2026
ft_junio = ft[
    (ft['PERIODO'] == 20260626) &
    (ft['FT'] == 1)
].copy()

desempleados_junio = desempleados[
    (desempleados['PERIODO'] == 20260626) &
    (desempleados['DSI'] == 1)
].copy()

# . Crear los grupos de edad
condiciones = [
    (ft_junio['P6040'] >= 18) & (ft_junio['P6040'] < 24),
    (ft_junio['P6040'] >= 24) & (ft_junio['P6040'] < 30),
    (ft_junio['P6040'] >= 30) & (ft_junio['P6040'] < 40),
    (ft_junio['P6040'] >= 40) & (ft_junio['P6040'] <= 50),
    (ft_junio['P6040'] >= 60)
]

categorias = [
    '18 a 24 años',
    '24 a 30 años',
    '30 a 40 años',
    '40 a 50 años',
    '60 años en adelante'
]

ft_junio['GRUPO_EDAD'] = np.select(
    condiciones,
    categorias,
    default='Fuera de los grupos'
)

# Crear los mismos grupos para los desempleados
condiciones_desempleados = [
    (desempleados_junio['P6040'] >= 18) & (desempleados_junio['P6040'] < 24),
    (desempleados_junio['P6040'] >= 24) & (desempleados_junio['P6040'] < 30),
    (desempleados_junio['P6040'] >= 30) & (desempleados_junio['P6040'] < 40),
    (desempleados_junio['P6040'] >= 40) & (desempleados_junio['P6040'] <= 50),
    (desempleados_junio['P6040'] >= 60)
]

desempleados_junio['GRUPO_EDAD'] = np.select(
    condiciones_desempleados,
    categorias,
    default='Fuera de los grupos'
)

# 3. Excluir las edades que no pertenecen a los grupos
ft_junio = ft_junio[
    ft_junio['GRUPO_EDAD'] != 'Fuera de los grupos'
].copy()

desempleados_junio = desempleados_junio[
    desempleados_junio['GRUPO_EDAD'] != 'Fuera de los grupos'
].copy()

# 4. Calcular la PEA por grupo de edad y departamento
pea_edad_dpto = ft_junio.groupby(
    ['GRUPO_EDAD', 'DPTO']
)['FEX_C18'].sum().rename('PEA')

# 5. Calcular los desempleados por grupo de edad y departamento
desempleados_edad_dpto = desempleados_junio.groupby(
    ['GRUPO_EDAD', 'DPTO']
)['FEX_C18'].sum().rename('DESEMPLEADOS')

# 6. Unir los resultados
tabla_desempleo_edad = pd.concat(
    [pea_edad_dpto, desempleados_edad_dpto],
    axis=1
).reset_index()

tabla_desempleo_edad['DESEMPLEADOS'] = (
    tabla_desempleo_edad['DESEMPLEADOS'].fillna(0)
)

# 7. Calcular la tasa de desempleo
tabla_desempleo_edad['TD'] = (
    tabla_desempleo_edad['DESEMPLEADOS'] /
    tabla_desempleo_edad['PEA']
) * 100

# 8. Asignar nombres a los departamentos
dpto_nombre = {
    5: 'ANTIOQUIA',
    8: 'ATLANTICO',
    11: 'BOGOTA',
    13: 'BOLIVAR',
    15: 'BOYACA',
    17: 'CALDAS',
    18: 'CAQUETA',
    19: 'CAUCA',
    20: 'CESAR',
    23: 'CORDOBA',
    25: 'CUNDINAMARCA',
    27: 'CHOCO',
    41: 'HUILA',
    44: 'GUAJIRA',
    47: 'MAGDALENA',
    50: 'META',
    52: 'NARIÑO',
    54: 'NORTE DE SANTANDER',
    63: 'QUINDIO',
    66: 'RISARALDA',
    68: 'SANTANDER',
    70: 'SUCRE',
    73: 'TOLIMA',
    76: 'VALLE DEL CAUCA',
    81: 'ARAUCA',
    85: 'CASANARE',
    86: 'PUTUMAYO',
    88: 'SAN ANDRES',
    91: 'AMAZONAS',
    94: 'GUAINIA',
    95: 'GUAVIARE',
    97: 'VAUPES',
    99: 'VICHADA'
}

tabla_desempleo_edad['Departamento'] = (
    tabla_desempleo_edad['DPTO'].map(dpto_nombre)
)

# 9. Mostrar la tabla completa
tabla_desempleo_edad = tabla_desempleo_edad.sort_values(
    ['Departamento', 'TD'],
    ascending=[True, False]
)

print(tabla_desempleo_edad)

# 10. Identificar la tasa más alta y la más baja
tabla_valida = tabla_desempleo_edad.dropna(
    subset=['Departamento', 'TD']
).copy()

indices_mayor = tabla_valida.groupby(
    'Departamento'
)['TD'].idxmax()

mayor_tasa = tabla_valida.loc[indices_mayor]

indices_menor = tabla_valida.groupby(
    'Departamento'
)['TD'].idxmin()

menor_tasa = tabla_valida.loc[indices_menor]



In [0]:
# 11. Mostrar los resultados
print("\nGRUPO DE EDAD CON MAYOR TASA DE DESEMPLEO:")
print(
    mayor_tasa[
        ['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']
    ].to_string(index=False)
)

print("\nGRUPO DE EDAD CON MENOR TASA DE DESEMPLEO:")
print(
    menor_tasa[
        ['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']
    ].to_string(index=False)
)


In [0]:
# PUNTO 3
# Grupo de edad con mayor y menor tasa de desempleo
# para cada departamento en junio de 2026

print("MAYOR TASA DE DESEMPLEO POR DEPARTAMENTO")

print(
    mayor_tasa[
        ['Departamento', 'GRUPO_EDAD', 'TD']
    ].to_string(index=False)
)

print("\nMENOR TASA DE DESEMPLEO POR DEPARTAMENTO")

print(
    menor_tasa[
        ['Departamento', 'GRUPO_EDAD', 'TD']
    ].to_string(index=False)
)

In [0]:
# Interpretación de los resultados – Punto 3

#En conclusión, este análisis permite reconocer que la tasa de desempleo no necesariamente es igual para todas las edades ni en todos los departamentos, por lo que resulta importante considerar las características de cada grupo poblacional al estudiar el mercado laboral colombiano.

In [0]:
#punto 4
# PUNTO 4
# Mujeres entre 18 y 25 años, junio de 2026

pea_mujeres = ft[
    (ft['PERIODO'] == 20260626) &
    (ft['FT'] == 1) &
    (ft['P3271'] == 2) &
    (ft['P6040'] >= 18) &
    (ft['P6040'] <= 25)
].groupby('DPTO')['FEX_C18'].sum()

desempleadas_mujeres = desempleados[
    (desempleados['PERIODO'] == 20260626) &
    (desempleados['DSI'] == 1) &
    (desempleados['P3271'] == 2) &
    (desempleados['P6040'] >= 18) &
    (desempleados['P6040'] <= 25)
].groupby('DPTO')['FEX_C18'].sum()

tabla_mujeres = pd.DataFrame({
    'PEA': pea_mujeres,
    'DESEMPLEADAS': desempleadas_mujeres
})

tabla_mujeres['DESEMPLEADAS'] = (
    tabla_mujeres['DESEMPLEADAS'].fillna(0)
)

tabla_mujeres['TD'] = (
    tabla_mujeres['DESEMPLEADAS'] /
    tabla_mujeres['PEA']
) * 100

tabla_mujeres['Departamento'] = (
    tabla_mujeres.index.map(dpto_nombre)
)

tres_menores = tabla_mujeres.dropna(
    subset=['Departamento', 'TD']
).sort_values('TD').head(3)

print(tres_menores[
    ['Departamento', 'PEA', 'DESEMPLEADAS', 'TD']
].to_string(index=False))

#  mujeres de 18 a 25 años en Vaupés (DPTO = 97)
#Significa que, en los datos analizados, no se registraron mujeres desempleadas de ese grupo de edad en Vaupés. pero no  quiere decir necesariamente que en la realidad ninguna mujer joven esté desempleada, sino que la estimación de esta muestra dio cero

In [0]:

#PUNTO 5
#Calcule el promedio de ingresos laborales (variable INGLABO) de las personas 
#residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle 
#del Cauca.
# Promedio de ingresos laborales por departamento

departamentos_seleccionados = {
    66: 'Risaralda',
    17: 'Caldas',
    63: 'Quindío',
    5: 'Antioquia',
    76: 'Valle del Cauca'
}

for codigo, nombre in departamentos_seleccionados.items():
    promedio = ocupados.loc[
        ocupados['DPTO'] == codigo, 'INGLABO'
    ].mean()

    print(f"{nombre}: ${promedio:,.2f}")


In [0]:

#PUNTO 5
# Promedio de ingresos laborales (INGLABO)
# Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca

promedio_INGLABO = ocupados[
    ocupados['DPTO'].isin([66, 17, 63, 5, 76])
]['INGLABO'].mean()

print("Promedio de ingresos laborales:", promedio_INGLABO)

#Antioquia registra el promedio de ingresos laborales más alto, con $2.754.449,60 COP.

#Valle del Cauca presenta el promedio más bajo, con $2.219.017,58 COP.

#Caldas alcanza un promedio de $2.427.893,31 COP, mientras que Quindío registra $2.289.058,93 COP.
#Risaralda presenta un promedio de $2.256.411,03 COP.

#el ingreso laboral promedio de las personas ocupadas de Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca es de aproximadamente $2.417.393 COP.
